In [ ]:
# Ability to automatically reload modules before executing code. This is useful for interactive development, as it allows you to modify your code and see the changes without restarting the kernel.
%load_ext autoreload
%autoreload 2

import os
import sys

# Add the parent directory to the Python path to allow importing modules from the src folder.
sys.path.append(os.path.abspath(os.path.join("..")))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# We do not import DataCleaner because EDA is performed on the raw data, before cleaning. However, we do import extract_age_in_days function from the preprocessing module, which is used to convert age to days.
from src.preprocessing import extract_age_in_days
from src.feature_engineering import extract_primary_breed, extract_primary_color

## EDA (Exploratory Data Analysis) ##
**Exploratory Data Analysis (EDA)** is the essential first step in the data science pipeline. Its primary purpose is to uncover the underlying structure of the dataset, investigate variables, and diagnose data quality before applying any predictive models. This is done by detecting missung values and outliers, and exploring distributions of individual features through statistical plots.



In [ ]:
df = pd.read_csv('data/raw_data/train.csv')

In [ ]:
df.info()

In [ ]:
counts = df['OutcomeType'].value_counts()
pct = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))

ax.set_title("Target Distribution (OutcomeType)", fontweight='bold')
ax.set_ylabel("Count")

offset_height = counts.max() * 0.02 

for bar, p in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset_height,
        f"{p:.1f}%",
        ha="center", 
        va="bottom", 
        fontsize=10
    )

plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


The 'OutcomeType' variable represents the target of the classification task. The distribution is not uniform, outcomes like 'Adoption' and 'Transfer' dominate,
while others are less represented. This indicates that the dataset was not obtained through stratified sampling and reflects real-world outcome frequencies at the Austin Animal Center.

In [ ]:
missing = df.isnull().sum()
print(missing)

In [ ]:
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values found in the dataset.")
else:
    
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot(kind="bar", ax=ax, color="salmon")
    
    ax.set_title("Missing Values by Column", fontweight='bold')
    ax.set_ylabel("Missing Values")
    
   
    offset_height = missing.max() * 0.02
    
    for i, v in enumerate(missing.values):
        ax.text(
            i, 
            v + offset_height, 
            str(v), 
            ha="center", 
            va="bottom", 
            fontsize=10
        )
        
    plt.tight_layout()
    plt.show()

**Handling Missing Values**

 'Name':
 We'll apply a univariate imputation strategy by replacing missing values (NaN) with the string 'Unknown'.
 This choice reflects that unnamed animals are likely strays or unowned, which can have behavioral or social implications.

'OutcomeSubtype':
 In principle, this variable could serve as an additional predictive target.
 However, due to the presence of nearly 50% missing entries, imputation is not appropriate
 and its inclusion would compromise data reliability.
 The feature is therefore excluded from further analysis to ensure consistent and interpretable results.

'SexuponOutcome':
 Only one value is missing in this feature.
 To balance functionality and statistical robustness, we replace the missing entry with the most frequent category (mode imputation).

'AgeuponOutcome':
 Only 18 values are missing in this feature.
 To balance functionality and statistical robustness, we replace the missing entries with the median of cats and dogs, respectively.

**AnimalType**

In [ ]:
feature= 'AnimalType' 
counts = df[feature].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))

ax.set_title(f"Distribution by {feature}", fontweight='bold')
ax.set_ylabel("Count")

# 4. Etichette di testo con offset dinamico
offset_height = counts.max() * 0.02

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset_height,
        f"{int(bar.get_height())}", 
        ha="center", 
        va="bottom", 
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
ct = pd.crosstab(df[feature], df['OutcomeType'], normalize="index")

fig, ax = plt.subplots(figsize=(10, 5))
ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")

ax.set_title(f"Outcome Distribution by {feature}", fontweight='bold')
ax.set_ylabel("Proportion")
ax.set_xlabel(feature)

plt.legend(title="Outcome", loc="upper right", bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.show()

When analyzing animal shelter data, looking at the dataset as a single aggregate entity can be misleading. 
For this reason, analysis is splitted from now on into two dedicated DataFrames: `df_dogs` and `df_cats`. 
By isolating the species, we can uncover distinct insights that would otherwise be lost in the noise of the global dataset.



**AgeuponOutcome**

In [ ]:
df_eda = df.copy()
df_eda["age_in_days"] = extract_age_in_days(df_eda["AgeuponOutcome"])
df_eda.drop(columns=["AgeuponOutcome"], inplace=True)

df_eda["DateTime"] = pd.to_datetime(df_eda["DateTime"])
df_eda["Month"] = df_eda["DateTime"].dt.month
df_eda["Hour"] = df_eda["DateTime"].dt.hour
df_eda["Weekday_Name"] = pd.Categorical(
    df_eda["DateTime"].dt.strftime("%a"),
    categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    ordered=True,
)

df_dogs_eda = df_eda[df_eda["AnimalType"] == "Dog"].copy()
df_cats_eda = df_eda[df_eda["AnimalType"] == "Cat"].copy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df_dogs_eda, x="age_in_days", ax=axes[0], color="teal")
axes[0].set_title("Boxplot of Age in Days (Dogs)")
axes[0].set_xlabel("Age (days)")

sns.histplot(
    data=df_dogs_eda,
    x="age_in_days",
    ax=axes[1],
    bins=50,
    color="teal"
)
axes[1].set_title("Distribution of Age in Days (Dogs)")
axes[1].set_xlabel("Age (days)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Percentiles of Age in Days:")
print(df_dogs_eda["age_in_days"].dropna().quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df_cats_eda, x="age_in_days", ax=axes[0], color="salmon")
axes[0].set_title("Boxplot of Age in Days (Cats)")
axes[0].set_xlabel("Age (days)")

sns.histplot(
    data=df_cats_eda,
    x="age_in_days",
    ax=axes[1],
    bins=50,
    color="salmon"
)
axes[1].set_title("Distribution of Age in Days (Cats)")
axes[1].set_xlabel("Age (days)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Percentiles of Age in Days:")
print(df_cats_eda["age_in_days"].dropna().quantile([0.5, 0.9, 0.95, 0.99, 0.999]))

**Age in Days (`age_in_days`) Distribution & Log Transformation**

* **Dogs:** The median age is 730 days (approx. 2 years), indicating a predominantly adult population navigating the shelter system.
* **Cats:** The median age drops drastically to just 90 days (approx. 3 months), underscoring the massive volume of kitten intakes compared to adult cats.
* **Handling Right-Skewness** Despite these different medians, both species share a common statistical trait: their age distributions are highly **right-skewed**. While the vast majority of animals are very young, there is a long, sparse tail of senior animals extending up to 6,205 days (approx. 17 years). The periodic spikes represent integer year boundaries (e.g., 365, 730 days) due to rounding in the original textual strings (e.g., "1 year", "2 years").
To mitigate the impact of this extreme skewness, we apply a **`log(x+1)` transformation** to the age feature in the preprocessing module. This mathematical transformation compresses the long tail and forces the data into a more normal (Gaussian) distribution. This is a crucial preprocessing step, as many Machine Learning algorithms (specifically **distance-based algorithms** (like KNN) and **linear models** (like Logistic Regression)) are highly sensitive to outliers and rely on normally distributed features to perform optimally and compute reliable weights. 
  

**Breed**

In [ ]:
feature = 'Breed'
top_n = 30 

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

counts_dogs = df_dogs_eda[feature].value_counts().head(top_n)
axes[0].bar(counts_dogs.index, counts_dogs.values, color="teal")
axes[0].set_title(f"Frequency of the First {top_n} Dog Breeds", fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=90) 

total_unique_dogs = df_dogs_eda[feature].nunique()
axes[0].text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique_dogs}", 
    transform=axes[0].transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)


counts_cats = df_cats_eda[feature].value_counts().head(top_n)

axes[1].bar(counts_cats.index, counts_cats.values, color="salmon")
axes[1].set_title(f"Frequency of the First {top_n} Cat Breeds", fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=90) 

total_unique_cats = df_cats_eda[feature].nunique()
axes[1].text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique_cats}", 
    transform=axes[1].transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)
plt.tight_layout()
plt.show()


**Color**

In [ ]:
feature = 'Color'
top_n = 30 

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

counts_dogs = df_dogs_eda[feature].value_counts().head(top_n)
axes[0].bar(counts_dogs.index, counts_dogs.values, color="teal")
axes[0].set_title(f"Frequency of the First {top_n} Dog Colors", fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=90) 

total_unique_dogs = df_dogs_eda[feature].nunique()
axes[0].text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique_dogs}", 
    transform=axes[0].transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)


counts_cats = df_cats_eda[feature].value_counts().head(top_n)

axes[1].bar(counts_cats.index, counts_cats.values, color="salmon")
axes[1].set_title(f"Frequency of the First {top_n} Cat Colors", fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=90) 

total_unique_cats = df_cats_eda[feature].nunique()
axes[1].text(
    0.95, 0.85, 
    f"Unique Categories: {total_unique_cats}", 
    transform=axes[1].transAxes, 
    ha='right', 
    va='top', 
    fontsize=11, 
    bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray')
)
plt.tight_layout()
plt.show()


**SexuponOutcome**

In [ ]:
df_cats_eda['SexuponOutcome'].value_counts(dropna=False)

In [ ]:
df_dogs_eda['SexuponOutcome'].value_counts(dropna=False)

In [ ]:
ct_sex_dogs = pd.crosstab(df_dogs_eda['SexuponOutcome'], df_dogs_eda['OutcomeType'], normalize='index')
ct_sex_cats = pd.crosstab(df_cats_eda['SexuponOutcome'], df_cats_eda['OutcomeType'], normalize='index')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ct_sex_dogs.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', legend=False)
axes[0].set_title("Outcome by Sex upon Outcome (Dogs)", fontweight='bold', fontsize=13)
axes[0].set_xlabel("Sex upon Outcome")
axes[0].set_ylabel("Proportion")

axes[0].tick_params(axis='x', rotation=45) 

ct_sex_cats.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', legend=False)
axes[1].set_title("Outcome by Sex upon Outcome (Cats)", fontweight='bold', fontsize=13)
axes[1].set_xlabel("Sex upon Outcome")
axes[1].tick_params(axis='x', rotation=45)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, title="OutcomeType", loc='center left', bbox_to_anchor=(0.92, 0.85), fontsize=11)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

**DateTime**

In [ ]:
ct_month = pd.crosstab(
    df_dogs_eda["Month"], df_dogs_eda["OutcomeType"], normalize="index"
)
ct_weekday = pd.crosstab(
    df_dogs_eda["Weekday_Name"], df_dogs_eda["OutcomeType"], normalize="index"
)
ct_hour = pd.crosstab(
    df_dogs_eda["Hour"], df_dogs_eda["OutcomeType"], normalize="index"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ct_month.plot(
    kind="bar", stacked=True, ax=axes[0], colormap="Set2", legend=False
)
axes[0].set_title("Outcome by Month (Dogs)", fontsize=14)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=90)

ct_weekday.plot(
    kind="bar", stacked=True, ax=axes[1], colormap="Set2", legend=False
)
axes[1].set_title("Outcome by Day of Week (Dogs)", fontsize=14)
axes[1].set_xlabel("Day")
axes[1].tick_params(axis="x", rotation=90)

ct_hour.plot(
    kind="bar", stacked=True, ax=axes[2], colormap="Set2", legend=False
)
axes[2].set_title("Outcome by Hour (Dogs)", fontsize=14)
axes[2].set_xlabel("Hour")
axes[2].tick_params(axis="x", rotation=90)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.91, 0.85),
    title="OutcomeType",
    fontsize=11,
)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
df_cats_eda["Weekday_Name"] = pd.Categorical(
    df_cats_eda["DateTime"].dt.strftime("%a"),
    categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    ordered=True,
)

ct_month = pd.crosstab(
    df_cats_eda["Month"], df_cats_eda["OutcomeType"], normalize="index"
)
ct_weekday = pd.crosstab(
    df_cats_eda["Weekday_Name"], df_cats_eda["OutcomeType"], normalize="index"
)
ct_hour = pd.crosstab(
    df_cats_eda["Hour"], df_cats_eda["OutcomeType"], normalize="index"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ct_month.plot(
    kind="bar", stacked=True, ax=axes[0], colormap="Set2", legend=False
)
axes[0].set_title("Outcome by Month (Cats)", fontsize=14)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=90)

ct_weekday.plot(
    kind="bar", stacked=True, ax=axes[1], colormap="Set2", legend=False
)
axes[1].set_title("Outcome by Day of Week (Cats)", fontsize=14)
axes[1].set_xlabel("Day")
axes[1].tick_params(axis="x", rotation=90)

ct_hour.plot(
    kind="bar", stacked=True, ax=axes[2], colormap="Set2", legend=False
)
axes[2].set_title("Outcome by Hour (Cats)", fontsize=14)
axes[2].set_xlabel("Hour")
axes[2].tick_params(axis="x", rotation=90)

handles, labels = axes[2].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.91, 0.85),
    title="OutcomeType",
    fontsize=11,
)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

## Feature Engineering ##
**Feature Engineering** is the process of using domain knowledge to transform raw data into features that better represent the underlying problem to the predictive models, thereby optimizing model accuracy and computational efficiency. It is done by enhancing current variables through mathematical transformations, converting qualitative or unstructured variables into machine-readable formats and generating brand-new features by combining or extracting information from existing columns.

In [ ]:
df_ml = df_eda.copy()

# --- Temporal Features ---
df_ml["IsWeekend_Label"] = np.where(df_ml["DateTime"].dt.dayofweek >= 5, "Weekend", "Weekday")
df_ml["TimeOfDay"] = pd.cut(
    df_ml["Hour"], 
    bins=[-1, 5, 11, 16, 23], 
    labels=["Night", "Morning", "Afternoon", "Evening"],
    right=True
).astype(str)

# --- Names ---
df_ml['clean_name'] = df_ml['Name'].fillna("").astype(str).str.strip()
df_ml['has_name'] = (df_ml['clean_name'].str.len() > 0).astype(int)

# --- Gender and Reproductive Status ---
is_male = df_ml["SexuponOutcome"].str.endswith("Male", na=False)
is_female = df_ml["SexuponOutcome"].str.endswith("Female", na=False)
df_ml["Gender"] = np.select([is_male, is_female], ["Male", "Female"], default=None)

is_neutered = df_ml["SexuponOutcome"].str.contains("Neutered|Spayed", na=False, case=False)
is_intact = df_ml["SexuponOutcome"].str.contains("Intact", na=False, case=False)
df_ml["Reproductive_Status"] = np.select([is_neutered, is_intact], ["Neutered/Spayed", "Intact"], default=None)

# --- Breed ---
is_mix = df_ml["Breed"].str.contains("Mix", na=False, case=False) | df_ml["Breed"].str.contains("/", na=False)
df_ml["Breed_Type"] = np.where(is_mix, "Mixed", "Purebred")
df_ml["Primary_Breed"] = extract_primary_breed(df_ml["Breed"])

# --- Color ---
df_ml["Is_Multicolor"] = np.where(df_ml["Color"].str.contains("/", na=False), "Multicolor", "Single Color")
df_ml["Primary_Color"] = extract_primary_color(df_ml["Color"])


df_dogs_ml = df_ml[df_ml["AnimalType"] == "Dog"].copy()
df_cats_ml = df_ml[df_ml["AnimalType"] == "Cat"].copy()

# Specific feature for cats
doy_cats = df_cats_ml["DateTime"].dt.dayofyear
df_cats_ml["KittenSeason_Label"] = np.where((doy_cats >= 91) & (doy_cats <= 304), "Kitten Season", "Out of Season")



columns_to_drop = [
    "Name",            
    "SexuponOutcome",  
    "Breed",           
    "Color",           
    "DateTime" 
]

df_dogs_ml.drop(columns=columns_to_drop, errors="ignore", inplace=True)
df_cats_ml.drop(columns=columns_to_drop, errors="ignore", inplace=True)

In [ ]:
ct_name_dogs = pd.crosstab(df_dogs_ml['has_name'], df_dogs_ml['OutcomeType'], normalize='index')
ct_name_cats = pd.crosstab(df_cats_ml['has_name'], df_cats_ml['OutcomeType'], normalize='index')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))


ct_name_dogs.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', legend=False)
axes[0].set_title("Impact of Name Presence on Outcome (Dogs)", fontweight='bold', fontsize=13)
axes[0].set_xlabel("Has a Name? (0 = No, 1 = Yes)")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis='x', rotation=0) # Mantiene 0 e 1 dritti

ct_name_cats.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', legend=False)
axes[1].set_title("Impact of Name Presence on Outcome (Cats)", fontweight='bold', fontsize=13)
axes[1].set_xlabel("Has a Name? (0 = No, 1 = Yes)")
axes[1].tick_params(axis='x', rotation=0)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, title="Outcome", loc='center left', bbox_to_anchor=(0.92, 0.85), fontsize=11)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)

ct_status_dogs = pd.crosstab(df_dogs_ml["Reproductive_Status"], df_dogs_ml["OutcomeType"], normalize="index")
ct_status_dogs.plot(kind="bar", stacked=True, ax=axes[0, 0], legend=False, colormap="Set2", width=0.5)
axes[0, 0].set_title("Outcome by Reproductive Status (Dogs)", fontsize=13)
axes[0, 0].set_xlabel("")
axes[0, 0].tick_params(axis="x", rotation=0)

ct_gen_dogs = pd.crosstab(df_dogs_ml["Gender"], df_dogs_ml["OutcomeType"], normalize="index")
ct_gen_dogs.plot(kind="bar", stacked=True, ax=axes[0, 1], legend=False, colormap="Set2", width=0.5)
axes[0, 1].set_title("Outcome by Gender (Dogs)", fontsize=13)
axes[0, 1].set_xlabel("")
axes[0, 1].tick_params(axis="x", rotation=0)

ct_status_cats = pd.crosstab(df_cats_ml["Reproductive_Status"], df_cats_ml["OutcomeType"], normalize="index")
ct_status_cats.plot(kind="bar", stacked=True, ax=axes[1, 0], legend=False, colormap="Set2", width=0.5)
axes[1, 0].set_title("Outcome by Reproductive Status (Cats)", fontsize=13)
axes[1, 0].set_xlabel("")
axes[1, 0].tick_params(axis="x", rotation=0)


ct_gen_cats = pd.crosstab(df_cats_ml["Gender"], df_cats_ml["OutcomeType"], normalize="index")
ct_gen_cats.plot(kind="bar", stacked=True, ax=axes[1, 1], legend=False, colormap="Set2", width=0.5)
axes[1, 1].set_title("Outcome by Gender (Cats)", fontsize=13)
axes[1, 1].set_xlabel("")
axes[1, 1].tick_params(axis="x", rotation=0)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.12, 0.8), title="OutcomeType")

plt.tight_layout()
plt.show()

In [ ]:
target_column = "Primary_Breed"
threshold_dogs = 0.003 

freq_dogs_original = df_dogs_ml[target_column].value_counts() / len(df_dogs_ml)
frequent_dogs = freq_dogs_original[freq_dogs_original >= threshold_dogs].index

df_dogs_ml["Breed_Binned_EDA"] = np.where(
    df_dogs_ml[target_column].isin(frequent_dogs) | df_dogs_ml[target_column].isna(),
    df_dogs_ml[target_column],
    "Other"
)
freq_dogs_binned = df_dogs_ml["Breed_Binned_EDA"].value_counts() / len(df_dogs_ml)


threshold_cats=0.005
freq_cats_original = df_cats_ml[target_column].value_counts() / len(df_cats_ml)
frequent_cats = freq_cats_original[freq_cats_original >= threshold_cats].index

df_cats_ml["Breed_Binned_EDA"] = np.where(
    df_cats_ml[target_column].isin(frequent_cats) | df_cats_ml[target_column].isna(),
    df_cats_ml[target_column],
    "Other"
)
freq_cats_binned = df_cats_ml["Breed_Binned_EDA"].value_counts() / len(df_cats_ml)


fig, axes = plt.subplots(1, 2, figsize=(18, 6))


sns.barplot(
    x=freq_dogs_binned.index, 
    y=freq_dogs_binned.values, 
    ax=axes[0], 
    hue=freq_dogs_binned.index,   
    palette="plasma",
    legend=False                  
)
axes[0].set_title(f"Binned {target_column} Frequencies - Dogs (0.3% Threshold)", fontsize=14)
axes[0].set_ylabel("Relative Frequency")
axes[0].set_xlabel("Breed Group")
axes[0].tick_params(axis="x", rotation=90)  # Risolve lo UserWarning di Matplotlib [3]


sns.barplot(
    x=freq_cats_binned.index, 
    y=freq_cats_binned.values, 
    ax=axes[1], 
    hue=freq_cats_binned.index,  
    palette="viridis",
    legend=False                 
)
axes[1].set_title(f"Binned {target_column} Frequencies - Cats (0.5% Threshold)", fontsize=14)
axes[1].set_ylabel("Relative Frequency")
axes[1].set_xlabel("Breed Group")
axes[1].tick_params(axis="x", rotation=90) 

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

ct_breed_dogs = pd.crosstab(df_dogs_ml["Breed_Type"], df_dogs_ml["OutcomeType"], normalize="index")

ct_breed_dogs.plot(kind="bar", stacked=True, ax=axes[0], legend=False,
                   colormap="Set2", width=0.4)
axes[0].set_title("Outcome by Breed Type (Dogs)", fontsize=14)
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)


ct_breed_cats = pd.crosstab(df_cats_ml["Breed_Type"], df_cats_ml["OutcomeType"], normalize="index")

ct_breed_cats.plot(kind="bar", stacked=True, ax=axes[1], legend=False,
                   colormap="Set2", width=0.4)
axes[1].set_title("Outcome by Breed Type (Cats)", fontsize=14)
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.12, 0.8), title="OutcomeType")

plt.tight_layout()
plt.show()

In [ ]:
target_column = "Primary_Color"
threshold = 0.015  


freq_original_dogs = df_dogs_ml[target_column].value_counts() / len(df_dogs_ml)
frequent_colors_dogs = freq_original_dogs[freq_original_dogs >= threshold].index

df_dogs_ml["Color_Binned_EDA"] = np.where(
    df_dogs_ml[target_column].isin(frequent_colors_dogs) | df_dogs_ml[target_column].isna(),
    df_dogs_ml[target_column],
    "Other"
)

freq_dogs_col_binned = df_dogs_ml["Color_Binned_EDA"].astype(str).value_counts() / len(df_dogs_ml)


freq_originale_cats = df_cats_ml[target_column].value_counts() / len(df_cats_ml)
frequent_colors_cats = freq_originale_cats[freq_originale_cats >= threshold].index

df_cats_ml["Color_Binned_EDA"] = np.where(
    df_cats_ml[target_column].isin(frequent_colors_cats) | df_cats_ml[target_column].isna(),
    df_cats_ml[target_column],
    "Other"
)

freq_cats_col_binned = df_cats_ml["Color_Binned_EDA"].astype(str).value_counts() / len(df_cats_ml)



fig, axes = plt.subplots(1, 2, figsize=(18, 6))


sns.barplot(
    x=freq_dogs_col_binned.index, 
    y=freq_dogs_col_binned.values, 
    ax=axes[0], 
    hue=freq_dogs_col_binned.index, 
    palette="plasma",
    legend=False                      
)
axes[0].set_title(f"Binned {target_column} - Dogs (1.5% Threshold)", fontsize=14)
axes[0].set_ylabel("Relative Frequency")
axes[0].set_xlabel("Color Group")
axes[0].tick_params(axis="x", rotation=90)  

sns.barplot(
    x=freq_cats_col_binned.index, 
    y=freq_cats_col_binned.values, 
    ax=axes[1], 
    hue=freq_cats_col_binned.index,   
    palette="viridis",
    legend=False                     
)
axes[1].set_title(f"Binned {target_column} - Cats (1.5% Threshold)", fontsize=14)
axes[1].set_ylabel("Relative Frequency")
axes[1].set_xlabel("Color Group")
axes[1].tick_params(axis="x", rotation=90)  

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)


ct_w_dogs = pd.crosstab(df_dogs_ml["IsWeekend_Label"], df_dogs_ml["OutcomeType"], normalize="index")
ct_w_dogs.plot(
    kind="bar", stacked=True, ax=axes[0], legend=False, colormap="Set2", width=0.5
)
axes[0].set_title("Outcome by Weekend (Dogs)", fontsize=14, fontweight='bold')
axes[0].set_xlabel("")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=0)


ct_w_cats = pd.crosstab(df_cats_ml["IsWeekend_Label"], df_cats_ml["OutcomeType"], normalize="index")
ct_w_cats.plot(
    kind="bar", stacked=True, ax=axes[1], legend=False, colormap="Set2", width=0.5
)
axes[1].set_title("Outcome by Weekend (Cats)", fontsize=14, fontweight='bold')
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)


handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.12, 0.8), title="OutcomeType")

plt.tight_layout()
plt.show()

In [ ]:
time_order = ["Night", "Morning", "Afternoon", "Evening"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

ct_t_dogs = pd.crosstab(df_dogs_ml["TimeOfDay"], df_dogs_ml["OutcomeType"], normalize="index").reindex(time_order)
ct_t_dogs.plot(
    kind="bar", stacked=True, ax=axes[0], legend=False,
    colormap="Set2", 
    width=0.6
)
axes[0].set_title("Outcome by Time of Day (Dogs)", fontsize=14)
axes[0].set_xlabel("Time of Day")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=0)

ct_t_cats = pd.crosstab(df_cats_ml["TimeOfDay"], df_cats_ml["OutcomeType"], normalize="index").reindex(time_order)
ct_t_cats.plot(
    kind="bar", stacked=True, ax=axes[1], legend=False,
    colormap="Set2", width=0.6
)
axes[1].set_title("Outcome by Time of Day (Cats)", fontsize=14)
axes[1].set_xlabel("Time of Day")
axes[1].tick_params(axis="x", rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.12, 0.8), title="OutcomeType")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct_k_cats = pd.crosstab(df_cats_ml["KittenSeason_Label"], df_cats_ml["OutcomeType"], normalize="index")
ct_k_cats.plot(
    kind="bar", stacked=True, ax=ax, legend=False,
    colormap="Set2", width=0.4
)

ax.set_title("Outcome by Kitten Season", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("Proportion")
ax.tick_params(axis="x", rotation=0)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="center left", bbox_to_anchor=(1.05, 0.8), title="OutcomeType")

plt.tight_layout()
plt.show()